# Проект 1. Влияние повышения цен на ПРАЙМ в mobile-сегменте

### Бизнес-вопрос: Было ли повышение цен на ПРАЙМ успешным для выручки с подписок mobile-сегмента с учётом изменения платёжного объёма?

In [ ]:
import os

from dotenv import find_dotenv, load_dotenv

# Находим .env в корне проекта
env_path = find_dotenv(usecwd=True)

if not env_path:
    raise SystemExit(
        "Файл .env не найден.\n"
        "Создайте его в корне репозитория рядом с README.md."
    )

# Загружаем переменные из .env
load_dotenv(env_path)

# Берём строку подключения
dsn = os.environ.get("PRIME_DSN")

if not dsn:
    raise SystemExit(
        f"Файл {env_path} найден, но переменной PRIME_DSN в нём нет."
    )

print("Файл .env найден")

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

engine = create_engine(dsn)

pd.read_sql(
    "select current_user, current_database(), version()",
    engine
).T

#### Определение сегмента и периода исследования

Mobile-сегмент — клиенты, у которых была хотя бы одна сессия с `device_type = 'mobile'`
в фиксированном периоде времени до повышения цен.

Состав сегмента фиксируется до изменения цены и затем не меняется при сравнении
периодов до и после.

Для сравнения используются два равных 28-дневных окна:
- до повышения цен: 01.02.2026–28.02.2026;
- после повышения цен: 01.03.2026–28.03.2026.

In [ ]:
# Период, по которому определяем mobile-сегмент

SEGMENT_START = "2026-01-01"
SEGMENT_END = "2026-03-01"

# Равные периоды для сравнения до и после повышения цены

BEFORE_START = "2026-02-01"
BEFORE_END = "2026-03-01"

AFTER_START = "2026-03-01"
AFTER_END = "2026-03-29"

DEVICE_TYPE = "mobile"

#### Формирование mobile-сегмента и характеристки клиентов

In [ ]:
# Находим клиентов, которые пользовались mobile
# до повышения цен

mobile_clients = pd.read_sql(
    text("""
        select distinct client_id
        from prime.service_usage
        where device_type = :device_type
          and started_at >= :segment_start
          and started_at < :segment_end
        order by client_id
    """),
    engine,
    params={
        "device_type": DEVICE_TYPE,
        "segment_start": SEGMENT_START,
        "segment_end": SEGMENT_END,
    }
)

print("ЗЕРНО: одна строка — один клиент.")
print("Клиентов в mobile-сегменте:", len(mobile_clients))
print("client_id уникальны:", mobile_clients["client_id"].is_unique)

mobile_clients.head()

In [ ]:
# Загружаем характеристики клиентов mobile-сегмента

clients = pd.read_sql(
    text("""
        select
            c.client_id,
            c.acquisition_channel,
            c.region_code
        from prime.clients c
        where c.client_id in (
            select distinct client_id
            from prime.service_usage
            where device_type = :device_type
              and started_at >= :segment_start
              and started_at < :segment_end
        )
        order by c.client_id
    """),
    engine,
    params={
        "device_type": DEVICE_TYPE,
        "segment_start": SEGMENT_START,
        "segment_end": SEGMENT_END,
    }
)

print("Строк:", len(clients))
print("Совпадает с mobile_clients:", len(clients) == len(mobile_clients))

clients.head()

### Подготовка данных о подписках и платежах

#### Подписки mobile-клиентов


In [ ]:
# Загружаем подписки, которыми владеют клиенты mobile-сегмента

subs = pd.read_sql(
    text("""
        select
            s.subscription_id,
            s.owner_client_id,
            s.tariff_id
        from prime.subscriptions s
        where s.owner_client_id in (
            select distinct client_id
            from prime.service_usage
            where device_type = :device_type
              and started_at >= :segment_start
              and started_at < :segment_end
        )
        order by s.subscription_id
    """),
    engine,
    params={
        "device_type": DEVICE_TYPE,
        "segment_start": SEGMENT_START,
        "segment_end": SEGMENT_END,
    }
)

# Проверяем зерно таблицы подписок
print("Подписок:", len(subs))
print("subscription_id уникальны:", subs["subscription_id"].is_unique)
print("Владельцев с подпиской:", subs["owner_client_id"].nunique())
print("owner_client_id уникальны:", subs["owner_client_id"].is_unique)

subs.head()

В mobile-сегменте 193 943 клиента, из них 153 595 владеют собственной подпиской. В выгрузке каждому владельцу соответствует ровно одна подписка: `subscription_id` и `owner_client_id` уникальны.

Таким образом, 40 348 клиентов mobile-сегмента не имеют собственной подписки. Для дальнейшего анализа выручки будем рассматривать платежи по 153 595 подпискам их владельцев, не размножая платежи на других пользователей подписки.

#### Успешные платежи до и после повышения цен

In [ ]:
# Загружаем успешные платежи по подпискам mobile-клиентов
# только за период исследования

pay = pd.read_sql(
    text("""
        select
            payment_id,
            subscription_id,
            paid_at,
            amount
        from prime.payments
        where status = 'success'
          and paid_at >= :before_start
          and paid_at < :after_end
          and subscription_id in (
              select subscription_id
              from prime.subscriptions
              where owner_client_id in (
                  select distinct client_id
                  from prime.service_usage
                  where device_type = :device_type
                    and started_at >= :segment_start
                    and started_at < :segment_end
              )
          )
        order by paid_at, payment_id
    """),
    engine,
    params={
        "before_start": BEFORE_START,
        "after_end": AFTER_END,
        "device_type": DEVICE_TYPE,
        "segment_start": SEGMENT_START,
        "segment_end": SEGMENT_END,
    }
)

# Выделяем календарный день платежа
pay["day"] = pay["paid_at"].dt.date

# Делим платежи на периоды до и после повышения цены
pay["period"] = "before"
pay.loc[
    pay["day"] >= pd.Timestamp(AFTER_START).date(),
    "period"
] = "after"

# Проверяем зерно: одна строка — один платеж
print("Платежей:", len(pay))
print("payment_id уникальны:", pay["payment_id"].is_unique)

# Проверяем, что в каждом окне есть все 28 дней
payment_check = (
    pay.groupby("period", as_index=False)
    .agg(
        n_payments=("payment_id", "size"),
        revenue=("amount", "sum"),
        n_days=("day", "nunique"),
    )
)

payment_check = payment_check.set_index("period").loc[["before", "after"]].reset_index()

payment_check

Оба периода содержат по 28 дней, их можно сравнивать напрямую. После повышения цен количество успешных платежей немного снизилось: с 102 267 до 101 436, при этом выручка выросла с 28,13 млн ₽ до 32,98 млн ₽.

Рост выручки произошёл несмотря на небольшое снижение платёжного объёма. Дальше нужно разделить этот эффект на изменение количества платежей и изменение среднего платежа.

#### Метрики до и после изменения цен по тарифам

In [ ]:
# Присоединяем к каждому платежу тариф подписки

pay_tariff = pay.merge(
    subs[["subscription_id", "tariff_id"]],
    on="subscription_id",
    how="left",
    validate="m:1"
)

# Проверяем, что merge не размножил платежи и деньги

print("Строк до merge:", len(pay))
print("Строк после merge:", len(pay_tariff))
print("Выручка до merge:", pay["amount"].sum())
print("Выручка после merge:", pay_tariff["amount"].sum())
print("Тариф не найден:", pay_tariff["tariff_id"].isna().sum())

In [ ]:
# Считаем основные метрики по тарифу и периоду

tariff_metrics = (
    pay_tariff
    .groupby(["tariff_id", "period"], as_index=False)
    .agg(
        revenue=("amount", "sum"),
        n_payments=("payment_id", "size")
    )
)

# Средний успешный платеж

tariff_metrics["avg_payment"] = (
    tariff_metrics["revenue"] / tariff_metrics["n_payments"]
)

# Для удобства сортируем: сначала before, потом after

period_order = pd.CategoricalDtype(
    categories=["before", "after"],
    ordered=True
)

tariff_metrics["period"] = tariff_metrics["period"].astype(period_order)

tariff_metrics = tariff_metrics.sort_values(
    ["tariff_id", "period"]
).reset_index(drop=True)

tariff_metrics

После повышения цен число успешных платежей немного снизилось на обоих тарифах, однако выручка выросла. У тарифа Basic количество платежей сократилось с 63 357 до 62 811, а средний успешный платёж вырос с 199 до 249 ₽. У Extended количество платежей снизилось с 38 910 до 38 625, а средний платёж вырос с 399 до 449 ₽.

Таким образом, предварительно рост выручки связан с увеличением суммы одного платежа, которое оказалось весомее небольшого снижения платёжного объёма.

#### Проверка изменения цен

In [ ]:
# Загружаем историю цен тарифов

tariffs = pd.read_sql(
    text("""
        select
            tariff_id,
            monthly_fee,
            valid_from,
            valid_to
        from prime.tariffs
        order by tariff_id, valid_from
    """),
    engine
)

tariffs

В справочнике тарифов обнаружена проблемная строка: для тарифа 2 цена 429 ₽ действует с 01.02.2026 по 30.06.2026 и пересекается с интервалами цен 399 ₽ и 449 ₽.

Такое пересечение означает, что одному платежу потенциально могут соответствовать сразу несколько цен.

In [ ]:
# Убираем ошибочную строку с ценой 429 ₽ (семинар 4)

tariffs_fixed = tariffs[tariffs["monthly_fee"] != 429].copy()

# Даты тарифов приводим к UTC,
# чтобы их можно было сравнивать с paid_at

# Приводим даты к единому формату UTC
for col in ["valid_from", "valid_to"]:
    tariffs_fixed[col] = pd.to_datetime(
        tariffs_fixed[col],
        utc=True
    )

tariffs_fixed

In [ ]:
# Соединяем платежи со всеми строками их тарифа

priced = pay_tariff.merge(
    tariffs_fixed[
        ["tariff_id", "monthly_fee", "valid_from", "valid_to"]
    ],
    on="tariff_id",
    how="inner",
    validate="m:m"
)

# Оставляем только ту цену, которая действовала
# в момент конкретного платежа

priced = priced[
    (priced["paid_at"] >= priced["valid_from"])
    & (
        priced["paid_at"]
        < priced["valid_to"] + pd.Timedelta(days=1)
    )
]

print("Платежей до:", len(pay_tariff))
print("Платежей с найденной ценой:", len(priced))
print("Выручка до:", pay_tariff["amount"].sum())
print("Выручка после:", priced["amount"].sum())
print(
    "Платежей, где amount не совпал с ценой:",
    (priced["amount"] != priced["monthly_fee"]).sum()
)

После исключения ошибочной строки с ценой 429 ₽ каждому из 203 703 платежей соответствует ровно одна действующая цена тарифа. Число платежей и общая выручка после присоединения справочника не изменились, а сумма каждого платежа совпадает с найденной тарифной ценой.

Дальнейшее изменение среднего платежа можно интерпретировать именно как эффект изменения цены тарифа, а не как результат ошибки соединения или структуры данных.

#### Факторная декомпозиция

In [ ]:
# Разделяем изменение выручки на два фактора:
# 1. изменение количества платежей
# 2. изменение среднего платежа

def chain(n0, p0, n1, p1):
    # Эффект изменения количества платежей
    volume_effect = (n1 - n0) * p0

    # Эффект изменения среднего платежа
    price_effect = n1 * (p1 - p0)

    # Проверяем, что два эффекта полностью объясняют изменение выручки
    revenue_change = n1 * p1 - n0 * p0

    assert abs(
        volume_effect + price_effect - revenue_change
    ) < 0.01

    return volume_effect, price_effect

In [ ]:
# Собираем результаты факторного анализа

decomposition = []

for tariff_id in tariff_metrics["tariff_id"].unique():
    t = tariff_metrics[tariff_metrics["tariff_id"] == tariff_id]

    before = t[t["period"] == "before"].iloc[0]
    after = t[t["period"] == "after"].iloc[0]

    volume_effect, price_effect = chain(
        before["n_payments"],
        before["avg_payment"],
        after["n_payments"],
        after["avg_payment"]
    )

    decomposition.append({
        "tariff_id": tariff_id,
        "revenue_before": before["revenue"],
        "revenue_after": after["revenue"],
        "revenue_change": after["revenue"] - before["revenue"],
        "volume_effect": volume_effect,
        "price_effect": price_effect
    })

decomposition = pd.DataFrame(decomposition)

decomposition

Факторная декомпозиция показывает, что рост выручки по обоим тарифам полностью обеспечен повышением цены, изменение платёжного объёма имело незначительное негативное влияние.

Для тарифа 1 снижение количества платежей уменьшило выручку на 108 654 ₽, но повышение цены добавило 3 140 550 ₽, поэтому итоговая выручка выросла на 3 031 896 ₽. Для тарифа 2 отрицательный эффект объёма составил 113 715 ₽, а положительный эффект цены — 1 931 250 ₽, что дало итоговый рост на 1 817 535 ₽.

### Предельно допустимое снижение платёжного объёма

In [ ]:
# Считаем фактическое и предельно допустимое
# снижение количества платежей по каждому тарифу

volume_limits = []

for tariff_id in tariff_metrics["tariff_id"].unique():
    t = tariff_metrics[tariff_metrics["tariff_id"] == tariff_id]

    before = t[t["period"] == "before"].iloc[0]
    after = t[t["period"] == "after"].iloc[0]

    # Фактическое снижение количества платежей
    actual_decline = (
        1 - after["n_payments"] / before["n_payments"]
    )

    # Максимальное снижение объёма,
    # при котором новая цена ещё сохраняет прежнюю выручку
    allowed_decline = (
        1 - before["avg_payment"] / after["avg_payment"]
    )

    # Запас до этой границы, в процентных пунктах
    reserve_pp = (
        allowed_decline - actual_decline
    ) * 100

    volume_limits.append({
        "tariff_id": tariff_id,
        "actual_decline_pct": actual_decline * 100,
        "allowed_decline_pct": allowed_decline * 100,
        "reserve_pp": reserve_pp
    })

volume_limits = pd.DataFrame(volume_limits)

volume_limits.round(2)

Фактическое снижение платёжного объёма оказалось значительно меньше уровня, который мог бы полностью компенсировать эффект повышения цены. Для тарифа 1 платежи снизились на 0,86% при допустимом снижении около 20,08%, для тарифа 2 — на 0,73% при допустимом снижении около 11,14%.

Таким образом, оба тарифа имеют заметный запас относительно границы, после которой повышение цены перестало бы поддерживать прежний уровень выручки.

#### Диагностика платёжного объёма по каналам привлечения

In [ ]:
# Добавляем владельца подписки к каждому платежу

pay_channel = pay_tariff.merge(
    subs[["subscription_id", "owner_client_id"]],
    on="subscription_id",
    how="left",
    validate="m:1"
)

# Добавляем канал привлечения владельца

pay_channel = pay_channel.merge(
    clients[["client_id", "acquisition_channel"]],
    left_on="owner_client_id",
    right_on="client_id",
    how="left",
    validate="m:1"
)

# Проверяем, что соединения не размножили платежи и деньги

print("Строк до:", len(pay_tariff))
print("Строк после:", len(pay_channel))

print("Выручка до:", pay_tariff["amount"].sum())
print("Выручка после:", pay_channel["amount"].sum())

print(
    "Платежей без известного канала:",
    pay_channel["acquisition_channel"].isna().sum()
)

In [ ]:
# Считаем платежи отдельно по тарифу, каналу и периоду

channel_payments = (
    pay_channel
    .groupby(
        ["tariff_id", "acquisition_channel", "period"],
        as_index=False,
        dropna=False
    )
    .agg(
        n_payments=("payment_id", "size")
    )
)

# Для удобства явно называем пропуски

channel_payments["acquisition_channel"] = (
    channel_payments["acquisition_channel"]
    .fillna("без канала")
)

# Задаём порядок периодов: сначала before, потом after

channel_payments["period"] = pd.Categorical(
    channel_payments["period"],
    categories=["before", "after"],
    ordered=True
)

# Сортируем таблицу
channel_payments = channel_payments.sort_values(
    ["tariff_id", "acquisition_channel", "period"]
).reset_index(drop=True)

channel_payments.head(30)

In [ ]:
# Ставим before и after рядом для каждого тарифа и канала

channel_change_base = (
    channel_payments
    .pivot(
        index=["tariff_id", "acquisition_channel"],
        columns="period",
        values="n_payments"
    )
    .reset_index()
)

channel_change_base.columns.name = None

# Явно задаём удобный порядок колонок
channel_change_base = channel_change_base[
    ["tariff_id", "acquisition_channel", "before", "after"]
]

# Считаем фактическое снижение платежей
# Положительное число = платежей стало меньше

channel_change_base["actual_decline_pct"] = (
    1
    - channel_change_base["after"] / channel_change_base["before"]
) * 100

channel_change_base

In [ ]:
# Добавляем предельно допустимое снижение
# для соответствующего тарифа

channel_change = channel_change_base.merge(
    volume_limits[
        ["tariff_id", "allowed_decline_pct"]
    ],
    on="tariff_id",
    how="left",
    validate="m:1"
)

# Считаем запас до границы
channel_change["reserve_pp"] = (
    channel_change["allowed_decline_pct"]
    - channel_change["actual_decline_pct"]
)

# Сначала показываем наиболее рискованные каналы

channel_change = (
    channel_change
    .sort_values(
        ["tariff_id", "reserve_pp"]
    )
    .reset_index(drop=True)
)

# Финальный порядок колонок
channel_change = channel_change[
    [
        "tariff_id",
        "acquisition_channel",
        "before",
        "after",
        "actual_decline_pct",
        "allowed_decline_pct",
        "reserve_pp"
    ]
]

channel_change.round(2)